# 03 — Spark Performance, Partitioning, Caching, and Cost

## The engineering question

“Partitioning is faster” and “cache improves performance” are incomplete answers.
Performance depends on query predicates, cardinality, file sizes, reuse, memory,
serialization, and shuffle boundaries. This lab collects evidence for each choice.

### Learning objectives

- Separate Spark partitions from storage partitions.
- Identify exchanges and scans in a physical plan.
- Measure file count, bytes, and repeated-action time.
- Demonstrate partition pruning.
- Compare repartition and coalesce.
- Use broadcast joins intentionally.
- Explain when caching helps and when it wastes memory.
- Convert bytes scanned into a configurable cost estimate.


In [1]:
from pathlib import Path
from time import perf_counter
import os
import shutil
import sys

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PERF_ROOT = PROJECT_ROOT / "lab_data" / "performance"

if PERF_ROOT.exists():
    assert (
        PERF_ROOT.name == "performance"
        and PROJECT_ROOT in PERF_ROOT.parents
    )
    shutil.rmtree(PERF_ROOT)

PERF_ROOT.mkdir(parents=True)

# Force Spark workers to use the same Python as this Jupyter kernel.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

try:
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql import types as T
except ImportError as exc:
    raise RuntimeError(
        "Install requirements-spark.txt before this lab"
    ) from exc

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("spark-performance-cost-lab")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.python.worker.reuse", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "16")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Notebook Python:", sys.executable)
print("Spark worker Python:", spark.sparkContext.pythonExec)


Notebook Python: C:\Users\sungj\sam_data_engineering_tutoring_capstone\.venv\Scripts\python.exe
Spark worker Python: C:\Users\sungj\sam_data_engineering_tutoring_capstone\.venv\Scripts\python.exe


## Generate a scalable workload without driver memory pressure

`spark.range()` creates data across Spark partitions. We avoid constructing a
million Python dictionaries on the driver. Increase `ROW_COUNT` only after the
baseline works on the student's machine.


```text
spark.range(200,000)
0부터 199,999까지 ID 생성
        ↓
event_id 생성
        ↓
customer_id 생성
        ↓
timestamp와 date 생성
        ↓
event_type 생성
├── view
├── cart
└── purchase
        ↓
region 생성
├── US-EAST
├── US-WEST
└── OTHER
        ↓
구매 event에 amount 추가
        ↓
임시 id 컬럼 삭제
        ↓
events DataFrame
총 20만 행
```

In [2]:
ROW_COUNT = 200_000  # Try 3–10 million on a machine with adequate memory.
INITIAL_PARTITIONS = max(4, spark.sparkContext.defaultParallelism * 2)
base_epoch = 1767225600  # 2026-01-01T00:00:00Z

events = (
    spark.range(ROW_COUNT, numPartitions=INITIAL_PARTITIONS)
    .withColumn("event_id", F.concat(F.lit("evt-"), F.lpad(F.col("id"), 10, "0")))
    .withColumn("customer_id", F.concat(F.lit("cust-"), F.lpad((F.col("id") % 50_000), 6, "0")))
    .withColumn("event_timestamp", F.timestamp_seconds(F.lit(base_epoch) + (F.col("id") % (30 * 86400))))
    .withColumn("event_date", F.to_date("event_timestamp"))
    .withColumn("event_type", F.expr("CASE WHEN id % 20 = 0 THEN 'purchase' WHEN id % 4 = 0 THEN 'cart' ELSE 'view' END"))
    .withColumn("region", F.expr("CASE WHEN id % 10 < 6 THEN 'US-EAST' WHEN id % 10 < 8 THEN 'US-WEST' ELSE 'OTHER' END"))
    .withColumn("amount", F.when(F.col("event_type") == "purchase", (F.col("id") % 200 + 1).cast("double")).otherwise(0.0))
    .drop("id")
)
print("Logical partitions:", events.rdd.getNumPartitions())
events.printSchema()


Logical partitions: 4
root
 |-- event_id: string (nullable = false)
 |-- customer_id: string (nullable = true)
 |-- event_timestamp: timestamp (nullable = true)
 |-- event_date: date (nullable = true)
 |-- event_type: string (nullable = false)
 |-- region: string (nullable = false)
 |-- amount: double (nullable = true)



## Storage layout experiment

Both datasets contain the same rows. One is a set of Parquet files in one
directory; the other uses Hive-style `event_date=...` prefixes. A date predicate
can prune unrelated storage partitions only in the second layout.


```text
events Spark DataFrame
20만 개의 합성 이벤트
        ↓
toPandas()
Windows에서 저장할 수 있도록 로컬 메모리로 변환
        ↓
동일한 데이터를 두 가지 방식으로 저장
        │
        ├── UNPARTITIONED
        │   날짜 구분 없이 하나의 Parquet 파일로 저장
        │
        │   events_unpartitioned/
        │   └── part-00000.parquet
        │
        └── PARTITIONED
            event_date별 폴더로 나누어 저장

            events_partitioned/
            ├── event_date=2026-01-01/
            ├── event_date=2026-01-02/
            └── event_date=2026-01-03/
        ↓
parquet_stats()
├── Parquet 파일 개수
├── 전체 파일 크기
└── 평균 파일 크기 계산
        ↓
layout_stats
두 저장 구조의 물리적 차이 비교
```
The data content is the same in both layouts. Only the physical storage organization is different.

The next experiment checks whether a date filter can read only the required partition instead of scanning the entire dataset.

In [3]:
UNPARTITIONED = PERF_ROOT / "events_unpartitioned"
PARTITIONED = PERF_ROOT / "events_partitioned"

import shutil

# Remove partial output left by the failed Spark write.
for output_path in (UNPARTITIONED, PARTITIONED):
    if output_path.exists():
        assert PERF_ROOT in output_path.parents
        shutil.rmtree(output_path)

# Spark created and transformed the dataset.
# PyArrow handles only the Windows local file-write boundary.
materialized_pdf = events.toPandas()
materialized_pdf["event_date"] = materialized_pdf["event_date"].astype(str)

# Unpartitioned layout
UNPARTITIONED.mkdir(parents=True, exist_ok=True)

materialized_pdf.to_parquet(
    UNPARTITIONED / "part-00000.parquet",
    index=False,
)

# Partitioned layout
materialized_pdf.to_parquet(
    PARTITIONED,
    partition_cols=["event_date"],
    index=False,
)

print("Windows fallback: PyArrow created both storage layouts.")


def parquet_stats(path: Path) -> dict:
    files = list(path.rglob("*.parquet"))
    total_bytes = sum(file.stat().st_size for file in files)

    return {
        "files": len(files),
        "total_mb": round(total_bytes / 1024**2, 2),
        "average_file_mb": round(
            total_bytes / max(len(files), 1) / 1024**2,
            2,
        ),
    }


layout_stats = {
    "unpartitioned": parquet_stats(UNPARTITIONED),
    "partitioned": parquet_stats(PARTITIONED),
}

layout_stats

Windows fallback: PyArrow created both storage layouts.


{'unpartitioned': {'files': 1, 'total_mb': 3.14, 'average_file_mb': 3.14},
 'partitioned': {'files': 3, 'total_mb': 4.17, 'average_file_mb': 1.39}}

동일한 날짜 데이터를 두 저장 구조에서 조회하고, 실제로 몇 개의 Parquet 파일을 읽는지 비교한다.

```text
저장된 event_date partition 검색
        ↓
실제로 존재하는 날짜 하나를 target_date로 선택
        ↓
동일한 날짜 조건 생성
event_date == target_date
        ↓
두 저장 구조에 같은 조건 적용
        │
        ├── UNPARTITIONED
        │   모든 날짜가 한 파일에 저장
        │   ↓
        │   전체 Parquet 파일을 확인
        │
        └── PARTITIONED
            날짜별 폴더에 저장
            ↓
            해당 날짜 partition만 선택
        ↓
PyArrow가 scan 정보 측정
├── all_fragments: 전체 Parquet 파일 수
├── selected_fragments: 조건에 맞아 읽은 파일 수
└── scan_seconds: 필터링하여 읽은 시간
        ↓
선택된 데이터를 Spark DataFrame으로 변환
        ↓
두 결과의 row count 비교
        ↓
행 수가 같으면 정확성 검증 통과
```

Expected result:

```text
Unpartitioned
all_fragments = 1
selected_fragments = 1

Partitioned
all_fragments = 3
selected_fragments = 1
```

Both queries return the same rows, but the partitioned layout can avoid reading unrelated date partitions. This behavior is called **partition pruning**.

In [4]:
def timed_count(frame, label: str) -> dict:
    started = perf_counter()
    rows = frame.count()
    elapsed = perf_counter() - started

    result = {
        "label": label,
        "rows": rows,
        "seconds": round(elapsed, 3),
    }

    print(result)
    return result


available_dates = sorted(
    path.name.split("=", 1)[1]
    for path in PARTITIONED.glob("event_date=*")
)

if not available_dates:
    raise RuntimeError(
        "No event_date partitions were found. Run the preceding write cell first."
    )

target_date = available_dates[len(available_dates) // 2]
print("Selected date with data:", target_date)

import pyarrow as pa
import pyarrow.dataset as ds


def pyarrow_date_scan(
    path: Path,
    *,
    hive_partitioned: bool,
):
    partitioning = (
        ds.partitioning(
            pa.schema([("event_date", pa.string())]),
            flavor="hive",
        )
        if hive_partitioned
        else None
    )

    dataset = ds.dataset(
        str(path),
        format="parquet",
        partitioning=partitioning,
    )

    predicate = ds.field("event_date") == target_date

    all_fragments = list(dataset.get_fragments())
    selected_fragments = list(
        dataset.get_fragments(filter=predicate)
    )

    started = perf_counter()
    table = dataset.to_table(filter=predicate)
    elapsed = perf_counter() - started

    evidence = {
        "all_fragments": len(all_fragments),
        "selected_fragments": len(selected_fragments),
        "scan_seconds": round(elapsed, 3),
    }

    # PyArrow performs the Windows file scan.
    # Spark continues processing the filtered result.
    spark_frame = spark.createDataFrame(table.to_pandas())

    return spark_frame, evidence


unpartitioned_read, unpartitioned_scan = pyarrow_date_scan(
    UNPARTITIONED,
    hive_partitioned=False,
)

partitioned_read, partitioned_scan = pyarrow_date_scan(
    PARTITIONED,
    hive_partitioned=True,
)

print("Unpartitioned scan evidence:", unpartitioned_scan)
print("Partitioned scan evidence:", partitioned_scan)

unpartitioned_result = timed_count(
    unpartitioned_read,
    "unpartitioned date filter",
)

partitioned_result = timed_count(
    partitioned_read,
    "partitioned date filter",
)

assert (
    unpartitioned_result["rows"]
    == partitioned_result["rows"]
)

print("\nPartitioned processing plan:")
partitioned_read.explain("formatted")

Selected date with data: 2026-01-02
Unpartitioned scan evidence: {'all_fragments': 1, 'selected_fragments': 1, 'scan_seconds': 0.01}
Partitioned scan evidence: {'all_fragments': 3, 'selected_fragments': 1, 'scan_seconds': 0.004}
{'label': 'unpartitioned date filter', 'rows': 86400, 'seconds': 0.594}
{'label': 'partitioned date filter', 'rows': 86400, 'seconds': 0.158}

Partitioned processing plan:
== Physical Plan ==
LocalTableScan (1)


(1) LocalTableScan
Output [7]: [event_id#15, customer_id#16, event_timestamp#17, event_type#18, region#19, amount#20, event_date#21]
Arguments: [event_id#15, customer_id#16, event_timestamp#17, event_type#18, region#19, amount#20, event_date#21]




```text
                    Unpartitioned      Partitioned
전체 파일 수              1                 3
선택한 파일 수            1                 1
조회 결과             86,400건          86,400건
PyArrow scan 시간       0.011초           0.004초
불필요한 날짜 제외        불가능             가능
```

### Read the plan, not just the stopwatch

Local timings vary because of operating-system cache, JIT warm-up, background
work, and file-system behavior. In the formatted plan, find `PartitionFilters`.
That is stronger evidence that Spark can skip unrelated date directories.

Partitioning has a cost: too many partition keys produce many directories and tiny
files. Choose keys used frequently in filters and avoid high-cardinality identifiers.


## Shuffle, repartition, and coalesce

- `repartition(n)` can increase or decrease partitions and normally performs a full
  shuffle.
- `coalesce(n)` usually reduces partitions without a full shuffle, but may create
  uneven work.
- `groupBy`, `distinct`, joins, and window functions commonly create exchanges.

Search the plans below for `Exchange`.


In [5]:
print("Partitions before:", events.rdd.getNumPartitions())
print("After repartition(24):", events.repartition(24).rdd.getNumPartitions())
print("After coalesce(4):", events.coalesce(4).rdd.getNumPartitions())

revenue_by_customer = events.groupBy("customer_id").agg(F.sum("amount").alias("revenue"))
revenue_by_customer.explain("formatted")

for shuffle_partitions in (4, 16, 64):
    spark.conf.set("spark.sql.shuffle.partitions", shuffle_partitions)
    result = timed_count(revenue_by_customer, f"groupBy with {shuffle_partitions} shuffle partitions")


Partitions before: 4
After repartition(24): 24
After coalesce(4): 4
== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Project (2)
            +- Range (1)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 200000, step=1, splits=Some(4))

(2) Project
Output [2]: [concat(cust-, lpad(cast((id#0L % 50000) as string), 6, 0)) AS customer_id#2, CASE WHEN CASE WHEN ((id#0L % 20) = 0) THEN true WHEN ((id#0L % 4) = 0) THEN false ELSE false END THEN cast(((id#0L % 200) + 1) as double) ELSE 0.0 END AS amount#7]
Input [1]: [id#0L]

(3) HashAggregate
Input [2]: [customer_id#2, amount#7]
Keys [1]: [customer_id#2]
Functions [1]: [partial_sum(amount#7)]
Aggregate Attributes [1]: [sum#53]
Results [2]: [customer_id#2, sum#54]

(4) Exchange
Input [2]: [customer_id#2, sum#54]
Arguments: hashpartitioning(customer_id#2, 16), ENSURE_REQUIREMENTS, [plan_id=144]

(5) HashAggregate
Input [2]: [customer_id#2, sum#54]
Keys [1]: [customer_id#

## Partition and Shuffle Experiment Results

### Partition Count Changes

```text
                        Original       repartition(24)       coalesce(4)
Partition 수               4                  24                   4
Partition 증가·감소         -                  증가                 감소
전체 Shuffle 발생           -                  발생              일반적으로 없음
주요 목적               초기 데이터        병렬 처리 증가       작은 파일 수 감소
```

```text
events
4 partitions
     ↓ repartition(24)
24 partitions
전체 데이터를 다시 분배
     ↓ coalesce(4)
4 partitions
인접 partition을 합쳐 개수를 감소
```

### GroupBy Shuffle Performance

```text
Shuffle partition 수            4               16               64
결과 customer 수             50,000           50,000           50,000
실행 시간                    0.135초           0.133초           0.212초
이번 실행의 결과             비슷함             가장 빠름           가장 느림
Partition 작업 크기          비교적 큼           적당함             너무 작음
관리해야 할 task 수             적음              적당함              많음
```

### Interpretation

```text
Partition이 너무 적으면
→ 각 task가 처리해야 할 데이터가 커질 수 있음

Partition 수가 적절하면
→ 여러 CPU가 데이터를 효율적으로 나누어 처리

Partition이 너무 많으면
→ 작은 task를 생성하고 관리하는 overhead가 증가
→ 현재처럼 작은 데이터에서는 오히려 느려질 수 있음
```

For this dataset, 16 shuffle partitions performed best, while 64 partitions added unnecessary task-management overhead. This does not mean 16 is always optimal; the result depends on data size, key distribution, available CPU, and cluster resources.

## Cache only reused, expensive intermediate results

Spark is lazy. `cache()` marks a DataFrame for persistence, but the first action
materializes it. A one-use DataFrame often becomes slower because cache population
itself costs CPU and memory. We reuse the same filtered relation twice.


In [6]:
purchases = events.filter(F.col("event_type") == "purchase")

uncached_a = timed_count(purchases.groupBy("region").count(), "uncached action 1")
uncached_b = timed_count(purchases.groupBy("event_date").count(), "uncached action 2")

cached_purchases = purchases.cache()
materialize = timed_count(cached_purchases, "cache materialization")
cached_a = timed_count(cached_purchases.groupBy("region").count(), "cached action 1")
cached_b = timed_count(cached_purchases.groupBy("event_date").count(), "cached action 2")
cached_purchases.unpersist()


{'label': 'uncached action 1', 'rows': 1, 'seconds': 0.207}
{'label': 'uncached action 2', 'rows': 3, 'seconds': 0.226}
{'label': 'cache materialization', 'rows': 10000, 'seconds': 0.377}
{'label': 'cached action 1', 'rows': 1, 'seconds': 0.157}
{'label': 'cached action 2', 'rows': 3, 'seconds': 0.147}


DataFrame[event_id: string, customer_id: string, event_timestamp: timestamp, event_date: date, event_type: string, region: string, amount: double]

```text
                                  Cache 없음        Cache 사용
지역별 집계 시간                     0.207초          0.157초
날짜별 집계 시간                     0.226초          0.147초
두 집계의 실행 시간 합계              0.433초          0.304초
Cache를 처음 만드는 시간                 -             0.377초
전체 시간                           0.433초          0.681초
```

## Translate scan reduction into a cost model

Query services such as Athena charge primarily by bytes scanned. The function below
is an educational estimate, not a pricing promise. Supply the vendor's current
price and use engine-reported scanned bytes when available.


In [7]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.capstone_utils import estimate_scan_cost

PRICE_PER_TB_EXAMPLE = 5.00
all_bytes = sum(file.stat().st_size for file in PARTITIONED.rglob("*.parquet"))
selected_partition = PARTITIONED / f"event_date={target_date}"
selected_bytes = sum(file.stat().st_size for file in selected_partition.rglob("*.parquet"))

cost_comparison = {
    "all_data_bytes": all_bytes,
    "selected_partition_bytes": selected_bytes,
    "estimated_full_scan_cost": estimate_scan_cost(all_bytes, PRICE_PER_TB_EXAMPLE),
    "estimated_pruned_scan_cost": estimate_scan_cost(selected_bytes, PRICE_PER_TB_EXAMPLE),
    "estimated_bytes_reduction_pct": round((1 - selected_bytes / all_bytes) * 100, 2),
}
cost_comparison


{'all_data_bytes': 4369611,
 'selected_partition_bytes': 1882466,
 'estimated_full_scan_cost': 1.9870690266543534e-05,
 'estimated_pruned_scan_cost': 8.560464266338386e-06,
 'estimated_bytes_reduction_pct': 56.92}

```text
파티셔닝 비용·성능 요약
────────────────────────────────────
전체 데이터 크기             4.37 MB
선택한 날짜 데이터           1.88 MB
읽지 않은 데이터             2.49 MB

실제로 읽은 비율              43.08%
읽기 데이터 감소율            56.92%

전체 Scan 예상 비용          $0.00001987
Partition Scan 예상 비용     $0.00000856
예상 비용 절감                $0.00001131
────────────────────────────────────
결론: 필요한 날짜 partition만 읽어
      데이터와 예상 비용을 약 57% 줄였다.

```

## Your turn

1. Partition by `customer_id` and document the small-file problem.
2. Create one customer responsible for 40% of events. Diagnose skew in the Spark UI.
3. Try salting the skewed join and compare correctness and plan complexity.
4. Run the cache experiment with only one action. Explain the result.
5. Record cold and warm runs separately in a Pandas results table.
6. Use the Spark UI SQL tab to capture scan, shuffle-read, and shuffle-write metrics.

### Interview checkpoint

Do not say “partitioning makes queries faster.” State the filter pattern, partition
cardinality, pruning evidence, file-count trade-off, and measured outcome.
